<a href="https://colab.research.google.com/github/hzhoujoy/ST554_HW6/blob/main/HW6_part_I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ST554_HomeWork_6   
Author: Huiping Zhou   
Date: 3/6/2026

# Part I-More Practice Querying a Database

We will continue working with the [lahman_1871-2022.sqlite](https://github.com/jknecht/baseball-archive-sqlite/releases/tag/2022) database, which contains information on Major League Baseball.

## Question 1
We will connect to the database and then look at all of the tables in the database by using `read_sql()` from pandas and returns it as a dataframe.

In [19]:
import sqlite3
import pandas as pd
# make the connection to the database
con = sqlite3.connect("lahman_1871-2022.sqlite")
#SQL query to return all table names in the data base
pd.read_sql("SELECT * FROM sqlite_master WHERE type='table';", con)

,type,name,tbl_name,rootpage,sql
0,table,AllstarFull,AllstarFull,2,"CREATE TABLE AllstarFull (\nplayerID TEXT,\nye..."
1,table,Appearances,Appearances,3,"CREATE TABLE Appearances (\nyearID INTEGER,\nt..."
2,table,AwardsManagers,AwardsManagers,4,"CREATE TABLE AwardsManagers (\nplayerID TEXT,\..."
3,table,AwardsPlayers,AwardsPlayers,5,"CREATE TABLE AwardsPlayers (\nplayerID TEXT,\n..."
4,table,AwardsShareManagers,AwardsShareManagers,6,CREATE TABLE AwardsShareManagers (\nawardID TE...
5,table,AwardsSharePlayers,AwardsSharePlayers,7,CREATE TABLE AwardsSharePlayers (\nawardID TEX...
6,table,Batting,Batting,8,"CREATE TABLE Batting (\nplayerID TEXT,\nyearID..."
7,table,BattingPost,BattingPost,9,"CREATE TABLE BattingPost (\nyearID INTEGER,\nr..."
8,table,CollegePlaying,CollegePlaying,10,"CREATE TABLE CollegePlaying (\nplayerID TEXT,\..."
9,table,Fielding,Fielding,11,"CREATE TABLE Fielding (\nplayerID TEXT,\nyearI..."


The lahman_1871-2022 database includes 27 tables.

## Question 2
Using SQL, construct a table of hall of fame pitchers (any hall of famer that pitched) that gives the playerID and their total (sum) for GS, G, W, L, IPOuts, CG, SHO, and SV columns. The summing can be done in pandas or in the SQL call.  

First of all, let us check a few rows of table Pitching

In [13]:
pd.read_sql('SELECT * FROM Pitching LIMIT 5;',con)

,playerID,yearID,stint,teamID,lgID,W,L,G,GS,CG,...,IBB,WP,HBP,BK,BFP,GF,R,SH,SF,GIDP
0,aardsda01,2004,1,SFN,NL,1,0,11,0,0,...,0,0,2,0,61,5,8,0,1,1
1,aardsda01,2006,1,CHN,NL,3,0,45,0,0,...,0,1,1,0,225,9,25,1,3,2
2,aardsda01,2007,1,CHA,AL,2,1,25,0,0,...,3,2,1,0,151,7,24,2,1,1
3,aardsda01,2008,1,BOS,AL,4,2,47,0,0,...,2,3,5,0,228,7,32,3,2,4
4,aardsda01,2009,1,SEA,AL,3,6,73,0,0,...,3,2,0,0,296,53,23,2,1,2


Then, we will construct a table named halloffame_pitchers by inner joining tje Pitching and HallOfFame on playerID. We filter to only only Hall of Fame inductees (inducted = 'Y') and return the columns playerID, GS, G, W, L, IPOuts, CG, SHO, and SV.

In [10]:
halloffame_pitchers = pd.read_sql("""
SELECT DISTINCT p.playerID, p.GS, p.G, p.W, p.L, p.IPOuts, p.CG, p.SHO, p.SV  FROM Pitching AS p
INNER JOIN HallOfFame AS h ON p.playerID = h.playerID
WHERE h.inducted = 'Y'
ORDER BY p.playerID;
""", con)
print(halloffame_pitchers.head())

    playerID  GS   G   W   L  IPouts  CG  SHO  SV
0  alexape01  37  48  28  13    1101  31    7   3
1  alexape01  34  46  19  17     931  25    3   3
2  alexape01  36  47  22   8     919  23    9   2
3  alexape01  39  46  27  15    1065  32    6   1
4  alexape01  42  49  31  10    1129  36   12   3


In [16]:
ct='''
    CREATE TABLE halloffame_pitchers AS
    SELECT DISTINCT p.playerID, p.GS, p.G, p.W, p.L, p.IPOuts, p.CG, p.SHO, p.SV  FROM Pitching AS p
    INNER JOIN HallOfFame AS h ON p.playerID = h.playerID
    WHERE h.inducted = 'Y'
    ORDER BY p.playerID;
'''
# Execute the CREATE TABLE statement
con.execute(ct)
# Commit the changes to the database
con.commit()
# Verify the table was created by querying it
pd.read_sql("SELECT * FROM halloffame_pitchers LIMIT 5;", con)

,playerID,GS,G,W,L,IPouts,CG,SHO,SV
0,alexape01,37,48,28,13,1101,31,7,3
1,alexape01,34,46,19,17,931,25,3,3
2,alexape01,36,47,22,8,919,23,9,2
3,alexape01,39,46,27,15,1065,32,6,1
4,alexape01,42,49,31,10,1129,36,12,3


In [17]:
summed = halloffame_pitchers.groupby("playerID").sum()
summed

,GS,G,W,L,IPouts,CG,SHO,SV
playerID,,,,,,,,
alexape01,599,696,373,208,15570,437,90,32
ansonca01,0,3,0,1,12,0,0,1
becklja01,1,1,0,1,12,0,0,0
bendech01,334,459,212,127,9051,255,40,34
blylebe01,685,692,287,250,14910,242,60,0
...,...,...,...,...,...,...,...,...
willivi01,471,513,249,205,11988,388,50,11
wrighge01,0,3,0,1,15,0,0,0
wrighha01,8,36,4,4,301,0,0,14
